# Asset Growth and the Cross-Section of Stock Returns
#### Asset Growth and the Cross-Section of Stock Returns (Cooper, Gulen & Schill, 2008)
#### Project Type: Academic replication and portfolio analysis

This notebook is a cleaned portfolio version. It is intended for demonstration purposes and does not redistribute restricted WRDS/CRSP/Compustat data.

## Portfolio note

This notebook presents a cleaned version of an academic replication project on the asset growth anomaly in cross-sectional stock returns.

**What this project shows**
- signal construction from Compustat fundamentals
- CRSP/Compustat linking through CCM
- descriptive diagnostics and stability checks
- portfolio sorts, long-short backtests, and FF3 risk adjustment

**Data access**
This repository does **not** redistribute WRDS, CRSP, or Compustat data. To run the notebook end to end, you need institutional access or locally cached input files placed in `data_raw/`.


## 1 Signal Replication：Asset Growth（Cooper/Gulen/Schill 2008）

This part replicates the paper’s Asset Growth signal and produces results on signal coverage, period-by-period cross-sectional descriptive statistics, and outlier sensitivity via winsorization.

### Prerequisites (cached files)

To avoid re-pulling from WRDS, please place these cached inputs locally:

- `data_raw/comp_funda_at.csv.gz` (Compustat `at` for signal construction)  
- `data_raw/ccm_linktable.csv.gz` (CCM mapping `gvkey` → `permno`)

If these files are missing, the notebook will attempt to download them from WRDS (requires access).

In [ ]:
# Install WRDS package if needed
# !pip install -q wrds

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from asset_growth import AssetGrowthSpec, apply_winsorization, compute_asset_growth_from_funda, link_gvkey_to_permno_ccm
from diagnostics import (
    cross_section_stats,
    coverage_by_security,
    coverage_by_time,
    panel_descriptive_stats,
    summarize_stats_over_time,
)
from wrds_utils import connect_wrds

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

DATA_RAW = ROOT / "data_raw"
DATA_INTERMEDIATE = ROOT / "data_intermediate"
RESULTS = ROOT / "results"

DATA_RAW.mkdir(exist_ok=True)
DATA_INTERMEDIATE.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

SPEC = AssetGrowthSpec(winsorize_by="formation_year", winsorize_p=0.01)

START_DATE = "1960-01-01"
END_DATE = None


In [ ]:
# 1) Compustat funda: total assets
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

comp_path = DATA_RAW / "comp_funda_at.csv.gz"

if comp_path.exists():
    comp = pd.read_csv(comp_path, compression="gzip", parse_dates=["datadate"], dtype={"gvkey": str})
    if "sic" not in comp.columns:
        comp["sic"] = np.nan
else:
    db = connect_wrds()

    sql_with_sic = """
        SELECT f.gvkey, f.datadate, f.at, c.sic
        FROM comp.funda AS f
        LEFT JOIN comp.company AS c
          ON f.gvkey = c.gvkey
        WHERE f.indfmt='INDL'
          AND f.datafmt='STD'
          AND f.consol='C'
          AND f.popsrc='D'
          AND f.at IS NOT NULL
          AND f.datadate >= %(start_date)s
    """

    sql_no_sic = """
        SELECT gvkey, datadate, at
        FROM comp.funda
        WHERE indfmt='INDL'
          AND datafmt='STD'
          AND consol='C'
          AND popsrc='D'
          AND at IS NOT NULL
          AND datadate >= %(start_date)s
    """

    try:
        comp = db.raw_sql(sql_with_sic, params={"start_date": START_DATE})
    except Exception:
        comp = db.raw_sql(sql_no_sic, params={"start_date": START_DATE})
        comp["sic"] = np.nan

    comp["datadate"] = pd.to_datetime(comp["datadate"])
    comp.to_csv(comp_path, index=False, compression="gzip")

comp.head(), comp.shape

In [ ]:
# 2) CCM linktable

ccm_path = DATA_RAW / "ccm_linktable.csv.gz"

if ccm_path.exists():
    ccm = pd.read_csv(
        ccm_path,
        compression="gzip",
        parse_dates=["linkdt", "linkenddt"],
        dtype={"gvkey": str},
    )
else:
    try:
        db
    except NameError:
        db = connect_wrds()

    ccm = db.raw_sql(
        """
        SELECT gvkey, lpermno, linkdt, linkenddt, linktype, linkprim
        FROM crsp.ccmxpf_linktable
        WHERE lpermno IS NOT NULL
        """
    )
    ccm["linkdt"] = pd.to_datetime(ccm["linkdt"])
    ccm["linkenddt"] = pd.to_datetime(ccm["linkenddt"])
    ccm.to_csv(ccm_path, index=False, compression="gzip")

ccm.head(), ccm.shape

In [ ]:
# 3) Calculate Asset Growth

sig = compute_asset_growth_from_funda(comp)

if "sic" not in sig.columns:
    sig["sic"] = np.nan
sig["sic"] = pd.to_numeric(sig["sic"], errors="coerce")

is_fin = sig["sic"].between(6000, 6999)
sig = sig[~is_fin].copy()

sig_linked = link_gvkey_to_permno_ccm(sig, ccm, prefer_primary=True)

sig_linked = apply_winsorization(sig_linked, spec=SPEC, value_col="asset_growth_raw")

sig_linked = sig_linked.replace([np.inf, -np.inf], np.nan)
sig_linked = sig_linked.dropna(subset=["permno", "formation_year", "asset_growth_raw", "asset_growth"])

signal_panel = sig_linked[[
    "permno",
    "gvkey",
    "datadate",
    "calyear",
    "formation_year",
    "formation_date",
    "sic",
    "asset_growth_raw",
    "asset_growth",
    "linktype",
    "linkprim",
]].copy()

signal_panel["permno"] = signal_panel["permno"].astype(int)

signal_panel.head(), signal_panel.shape

In [ ]:
# 4) Coverage and Descriptive Statistics

cov_year = coverage_by_time(signal_panel, time_col="formation_year", id_col="permno")
cov_sec = coverage_by_security(signal_panel, time_col="formation_year", id_col="permno")

stats_by_year = panel_descriptive_stats(signal_panel, time_col="formation_year", value_col="asset_growth")
stats_summary = summarize_stats_over_time(stats_by_year, time_col="formation_year")

cov_year.head(), stats_by_year.head(), cov_sec.head()

In [ ]:
# 4c) Stability: Summarize coverage and distribution by subsamples (by decade).

signal_panel["decade"] = (signal_panel["formation_year"] // 10) * 10

decade_cov = (
    signal_panel.groupby("decade")["permno"].nunique().reset_index(name="n_securities")
)

decade_stats = (
    signal_panel.groupby("decade")["asset_growth"]
    .apply(lambda s: pd.Series({
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "p05": s.quantile(0.05),
        "p50": s.quantile(0.50),
        "p95": s.quantile(0.95),
    }))
    .reset_index()
)

decade_cov, decade_stats

In [ ]:
# 4b) winsorize：raw vs winsorized

stats_by_year_raw = panel_descriptive_stats(signal_panel, time_col="formation_year", value_col="asset_growth_raw")

clip_rate_by_year = (
    (signal_panel["asset_growth"] != signal_panel["asset_growth_raw"])  # 被 clip 的占比
    .groupby(signal_panel["formation_year"])
    .mean()
    .reset_index(name="winsorized_clip_rate")
)

raw_vs_w = stats_by_year_raw.merge(
    stats_by_year[["formation_year", "p01", "p99"]].rename(columns={"p01": "p01_w", "p99": "p99_w"}),
    on="formation_year",
    how="left",
)

clip_rate_by_year.head(), raw_vs_w.head()

In [ ]:
# 5) intermediate + results

signal_out = DATA_INTERMEDIATE / "asset_growth_signal.csv.gz"
coverage_out = RESULTS / "coverage_by_year.csv"
stats_by_year_out = RESULTS / "asset_growth_stats_by_year.csv"
stats_by_year_raw_out = RESULTS / "asset_growth_stats_by_year_raw.csv"
clip_rate_by_year_out = RESULTS / "asset_growth_winsor_clip_rate_by_year.csv"
stats_summary_out = RESULTS / "asset_growth_stats_summary_over_time.csv"
coverage_by_security_out = RESULTS / "coverage_by_security.csv"

signal_panel.to_csv(signal_out, index=False, compression="gzip")
cov_year.to_csv(coverage_out, index=False)
stats_by_year.to_csv(stats_by_year_out, index=False)
stats_by_year_raw.to_csv(stats_by_year_raw_out, index=False)
clip_rate_by_year.to_csv(clip_rate_by_year_out, index=False)
stats_summary.to_csv(stats_summary_out, index=False)
cov_sec.to_csv(coverage_by_security_out, index=False)

print("Wrote:")
print("-", signal_out)
print("-", coverage_out)
print("-", stats_by_year_out)
print("-", stats_by_year_raw_out)
print("-", clip_rate_by_year_out)
print("-", stats_summary_out)
print("-", coverage_by_security_out)

print("Signal panel years:", int(signal_panel["formation_year"].min()), "to", int(signal_panel["formation_year"].max()))
print("N formation_years:", signal_panel["formation_year"].nunique())
print("N permno:", signal_panel["permno"].nunique())

In [ ]:
# 6) full sample

overall_w = cross_section_stats(signal_panel["asset_growth"])
overall_raw = cross_section_stats(signal_panel["asset_growth_raw"])

overall_df = pd.DataFrame([
    {"variant": "winsorized", **overall_w},
    {"variant": "raw", **overall_raw},
])

overall_out = RESULTS / "asset_growth_stats_full_sample.csv"
overall_df.to_csv(overall_out, index=False)
print("Wrote:", overall_out)
overall_df

## 2 Diagnostics & Signal Validation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(DATA_INTERMEDIATE / "asset_growth_signal.csv.gz", compression="gzip")
df["asset_growth_raw"] = df["asset_growth_raw"].clip(-5,5)

#Convert Time Variable
df["formation_date"] = pd.to_datetime(df["formation_date"])
df = df.sort_values(["formation_date","permno"])
df = df.reset_index(drop=True)

In [ ]:
signal_rep = "asset_growth_raw"
signal_wrds = "asset_growth"
time_col = "formation_date"

### 2.1 Cross-sectional descriptive statistics

In [ ]:
#Cross-Sectional Descriptive Statistics (Each Date)
def cross_section_stats(x):

    return pd.Series({
        "count": x.count(),
        "mean": x.mean(),
        "median": x.median(),
        "std": x.std(),
        "min": x.min(),
        "p1": x.quantile(0.01),
        "p5": x.quantile(0.05),
        "p25": x.quantile(0.25),
        "p75": x.quantile(0.75),
        "p95": x.quantile(0.95),
        "p99": x.quantile(0.99),
        "max": x.max(),
        "skew": x.skew(),
        "kurtosis": x.kurt()
    })

cs_stats = df.groupby(time_col)[signal_rep].apply(cross_section_stats).reset_index()

In [ ]:
# --- A. Compute Cross-sectional Statistics by Year ---
annual_stats = df.groupby('formation_year')['asset_growth'].agg([
    'count',   # Number of observations
    'mean',    # Average
    'std',     # Standard Deviation
    'min',     # Minimum
    lambda x: x.quantile(0.25), # 25th Percentile
    'median',  # Median (50th Percentile)
    lambda x: x.quantile(0.75), # 75th Percentile
    'max'      # Maximum
]).rename(columns={'<lambda_0>': '25%', '<lambda_1>': '75%'})

# --- B. Summarized Over Time ---
# Calculate the average of these annual statistics across all years
time_series_summary = annual_stats.mean().to_frame(name='Average Annual Stats')

# --- C. Full Sample Statistics ---
full_sample_summary = df['asset_growth'].describe().to_frame(name='Full Sample Stats')

print("--- Table 1: Average of Annual Cross-sectional Statistics ---")
print(time_series_summary)
print("\n--- Table 2: Overall Full Sample Statistics ---")
print(full_sample_summary)

# --- D. Visualization: Signal Trend Over Time ---
plt.figure(figsize=(10, 5))
plt.plot(annual_stats.index, annual_stats['mean'], label='Mean')
plt.plot(annual_stats.index, annual_stats['median'], label='Median', linestyle='--')
plt.title('Asset Growth Signal Trend Over Time')
plt.xlabel('Formation Year')
plt.ylabel('Signal Value')
plt.legend()
plt.grid(True)
plt.savefig('asset_growth_trend.png') # Saving as a file

In [ ]:
#Summarize Cross-Sectional Statistics Over Time
summary_over_time = cs_stats.drop(columns=time_col).describe().T

summary_over_time = summary_over_time[
    ["mean","std","min","25%","50%","75%","max"]
]

print(summary_over_time)

In [ ]:
#Full Sample Descriptive Statistics
full_sample = df[signal_rep].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99])

skew = df[signal_rep].skew()
kurt = df[signal_rep].kurt()

print(full_sample)
print("Skewness:",skew)
print("Kurtosis:",kurt)

In [ ]:
#Distribution Plot
plt.figure(figsize=(8,5))
sns.histplot(df[signal_rep], bins=100, kde=True)
plt.title("Distribution of Asset Growth Signal")
plt.xlabel("Asset Growth")
plt.show()

### 2.2 Outlier and stability analysis

In [ ]:
#Outlier Analysis
def outlier_share(x):

    p1 = x.quantile(0.01)
    p99 = x.quantile(0.99)

    lower = (x < p1).mean()
    upper = (x > p99).mean()

    return pd.Series({
        "lower_outliers": lower,
        "upper_outliers": upper
    })

outliers = df.groupby(time_col)[signal_rep].apply(outlier_share)

print(outliers.describe())

In [ ]:
#Winsorization Test
def winsorize(series, lower=0.01, upper=0.99):

    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)

    return series.clip(lower_bound, upper_bound)

df["signal_wins"] = df.groupby(time_col)[signal_rep].transform(winsorize)
#Compare raw vs winsorized.
comparison_stats = pd.DataFrame({
    "raw_mean": df[signal_rep].mean(),
    "wins_mean": df["signal_wins"].mean(),
    "raw_std": df[signal_rep].std(),
    "wins_std": df["signal_wins"].std()
}, index=[0])

print(comparison_stats)

In [ ]:
#Stability Analysis
mean_ts = df.groupby(time_col)[signal_rep].mean()
std_ts = df.groupby(time_col)[signal_rep].std()

plt.figure(figsize=(10,4))
mean_ts.plot()
plt.title("Cross-Sectional Mean Over Time")
plt.show()

plt.figure(figsize=(10,4))
std_ts.plot()
plt.title("Cross-Sectional Std Over Time")
plt.show()

In [ ]:
# Ensure data is sorted by firm (permno) and time (formation_year)
df_sorted = df.sort_values(['permno', 'formation_year'])

# Create a lagged signal (Asset Growth from the previous year)
df_sorted['asset_growth_lag'] = df_sorted.groupby('permno')['asset_growth'].shift(1)

# Calculate Rank Correlation (Spearman) to measure signal persistence
# A higher coefficient indicates the signal is more stable over time
persistence = df_sorted[['asset_growth', 'asset_growth_lag']].corr(method='spearman').iloc[0, 1]

print(f"Signal Stability (Year-over-Year Rank Correlation): {persistence:.4f}")

In [ ]:
#Signal Persistence
df["lag_signal"] = df.groupby("permno")[signal_rep].shift(1)

lag_corr = df.groupby(time_col).apply(
    lambda x: x[signal_rep].corr(x["lag_signal"])
)

print("Average lag correlation:",lag_corr.mean())

### 2.3 Replicated signal vs the prebuilt WRDS signal

In [ ]:
import wrds
import pandas as pd

db = wrds.Connection()

signal_wrds = db.raw_sql("""
SELECT
    permno,
    fdate AS date,
    atg
FROM wrdsapps.signals_raw_plus
WHERE fdate BETWEEN '2005-01-01' AND '2020-12-31'
""")

In [ ]:
rep_df = df[["permno", "formation_date", "asset_growth_raw"]].copy()

rep_df = rep_df.rename(columns={
    "formation_date": "date",
    "asset_growth_raw": "ag_rep"
})
wrds_df = signal_wrds.rename(columns={
    "atg": "ag_wrds"
})

In [ ]:

rep_df["date"] = pd.to_datetime(rep_df["date"])
wrds_df["date"] = pd.to_datetime(wrds_df["date"])
rep_df["ag_rep"] = -rep_df["ag_rep"]
df_compare = pd.merge(
    rep_df,
    wrds_df,
    on=["permno", "date"],
    how="inner"
)

df_compare.head()

In [ ]:
df_compare[["ag_rep","ag_wrds"]].corr()

In [ ]:
df_compare["diff"] = df_compare["ag_rep"] - df_compare["ag_wrds"]
df_compare["diff"].describe()

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(x="ag_rep", y="ag_wrds", data=df_compare, alpha=0.6)

plt.title("Replicated Asset Growth vs WRDS Asset Growth", fontsize=14)
plt.xlabel("Replicated Asset Growth Signal", fontsize=12)
plt.ylabel("WRDS Asset Growth Signal", fontsize=12)

plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
corr = df_compare.groupby("date").apply(
    lambda x: x["ag_rep"].corr(x["ag_wrds"])
)

corr.describe()

In [ ]:
rep_df["ag_rep"].describe()

The average difference is very close to zero at **-0.024**, while the standard deviation of the difference is around **0.48**, suggesting that most differences are relatively small.

Overall, these results indicate that our replicated asset growth signal closely matches the WRDS version. Any remaining differences are likely due to **minor variations in data cleaning, timing conventions, or accounting data processing**.

This comparison gives us confidence that our signal construction is reliable and suitable for the subsequent portfolio analysis.

## 3 Portfolio Evaluation

In [ ]:
df = pd.read_csv(DATA_INTERMEDIATE / "asset_growth_signal.csv.gz", compression="gzip")
df.head(3)

In [ ]:
import wrds
db = wrds.Connection()
msf = db.raw_sql("select permno, date, ret, prc, shrout from crsp.msf")
msenames = db.raw_sql("select permno, namedt, nameendt, shrcd from crsp.msenames")

In [ ]:
# CRSP data cleaning
msf["date"] = pd.to_datetime(msf["date"])
msenames["namedt"] = pd.to_datetime(msenames ["namedt"])
msenames["nameendt"] = pd.to_datetime(msenames["nameendt"])

df = msf.merge(msenames, on="permno", how="inner")
df = df[(df["date"] >= df["namedt"]) & (df["date"] <= df["nameendt"])]
df["date"] = df["date"] + pd.offsets.MonthEnd(0)
df = df[df["shrcd"].isin([10,11])]

df["mkt_cap"] = df["prc"].abs() * df["shrout"] / 1000
df = df[["permno", "date", "ret", "mkt_cap"]].copy()
df = df.sort_values(["permno","date"])

df["ret"] = pd.to_numeric(df["ret"], errors="coerce")
df = df.dropna(subset=["ret"])

print(df.shape)

In [ ]:
sig = pd.read_csv("asset_growth_signal.csv.gz", compression="gzip", parse_dates=["formation_date"])
sig = sig[["permno","formation_date","asset_growth"]]

In [ ]:
# Backtest window: July 2009 – June 2024
# Formation window: June 2009 - June 2023
sig = sig[(sig["formation_date"] >= "2009-06-30") & (sig["formation_date"] <= "2023-06-30")].copy()
sig['hold_start'] = sig['formation_date'] + pd.offsets.MonthEnd(1)
sig['hold_end'] = sig['formation_date'] + pd.DateOffset(months=12)
# Return window: July 2009 - June 2024
df = df[(df["date"] >= "2009-07-31") & (df["date"] <= "2024-06-30")].copy()

In [ ]:
# Merge signal and CRSP returns & keep holding period
panel = sig.merge(df, on="permno", how="inner")
panel = panel[(panel["date"] >= panel["hold_start"]) & (panel["date"] <= panel["hold_end"])].copy()
print(panel.shape)

In [ ]:
panel.head(3)

### 3.1 Equal-weighted and value-weighted quintile or decile portfolios

In [ ]:
# Decile portfolios
assign = (
    panel[["permno", "formation_date", "asset_growth"]]
    .drop_duplicates()
    .copy()
)

assign["decile"] = (
    assign.groupby("formation_date")["asset_growth"]
    .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates="drop"))
)

assign["decile"] = assign["decile"] + 1

panel = panel.drop(columns="decile", errors="ignore").merge(
    assign,
    on=["permno", "formation_date", "asset_growth"],
    how="left"
)

In [ ]:
panel.head(3)

In [ ]:
# Equal-weighted returns
ew = panel.groupby(["date","decile"])["ret"].mean().reset_index()

ew = ew.pivot(index="date", columns="decile", values="ret")

In [ ]:
# Value-weighted returns
panel["weight"] = panel.groupby(["date","decile"])["mkt_cap"]\
    .transform(lambda x: x / x.sum())

panel["vw_ret"] = panel["weight"] * panel["ret"]

vw = panel.groupby(["date","decile"])["vw_ret"].sum().reset_index()

vw = vw.pivot(index="date", columns="decile", values="vw_ret")

### 3.2 Long-short returns, cumulative returns and cumulative risk-adjusted performance

In [ ]:
# Long-short portfolio
ew["LS"] = ew[1] - ew[10]
vw["LS"] = vw[1] - vw[10]

In [ ]:
# Cumulative return
cum_ew = (1 + ew["LS"]).cumprod() - 1
cum_vw = (1 + vw["LS"]).cumprod() - 1

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.plot(cum_ew, label="EW Long-Short", color="darkred", linewidth=1)
plt.plot(cum_vw, label="VW Long-Short", color="steelblue", linewidth=1)

plt.legend()
plt.title("Asset Growth Long-Short Cumulative Return")
plt.show()

The cumulative return plot illustrates that the long-short strategy generates persistent performance over the full backtesting period. **The equal-weighted strategy produces stronger long-term growth**, while the value-weighted strategy shows higher volatility.

In [ ]:
# Performance matrix
import numpy as np

def perf_stats(r: pd.Series) -> pd.Series:
    r = pd.to_numeric(r, errors="coerce").dropna()

    mean_monthly = r.mean()
    ann_return = (1 + mean_monthly) ** 12 - 1
    ann_vol = r.std(ddof=1) * np.sqrt(12)
    sharpe = mean_monthly / r.std(ddof=1) * np.sqrt(12)

    wealth = (1 + r).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    max_dd = drawdown.min()

    t_stat = mean_monthly / (r.std(ddof=1) / np.sqrt(len(r)))

    return pd.Series({
        "Mean Monthly": mean_monthly,
        "Annual Return": ann_return,
        "Annual Vol": ann_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_dd,
        "t-stat": t_stat,
        "N Months": len(r),
    })

perf_table = pd.DataFrame({
    "EW Long-Short": perf_stats(ew["LS"]),
    "VW Long-Short": perf_stats(vw["LS"]),
}).T

perf_table

The asset growth anomaly generates economically large and statistically significant returns.

The equal-weighted long-short portfolio delivers an annual return of about 15% with a Sharpe ratio close to 1 and a t-statistic of 3.77.

The value-weighted strategy is weaker, suggesting that the anomaly is stronger among smaller firms.

### 3.3 Risk-Adjusted Performance -- FF3 Regression

In [ ]:
ff = db.raw_sql("select date, mktrf, smb, hml, rf from ff.factors_monthly")

In [ ]:
ff["date"] = pd.to_datetime(ff["date"])
ff[["mktrf","smb","hml","rf"]] = ff[["mktrf","smb","hml","rf"]].astype(float)

In [ ]:
print(ff.head())
print(ff.describe())
print(ff.dtypes)

In [ ]:
# FF3 regression
import statsmodels.api as sm

def run_ff3(ls_series: pd.Series, ff_df: pd.DataFrame, maxlags: int = 6):
    ls = ls_series.reset_index().copy()
    ls.columns = ["date", "LS"]
    ls["date"] = pd.to_datetime(ls["date"])
    ls["ym"] = ls["date"].dt.to_period("M")

    reg = ls.merge(
        ff_df[["ym", "mktrf", "smb", "hml", "rf"]],
        on="ym",
        how="inner"
    ).copy()

    reg["LS"] = pd.to_numeric(reg["LS"], errors="coerce")
    reg["mktrf"] = pd.to_numeric(reg["mktrf"], errors="coerce")
    reg["smb"] = pd.to_numeric(reg["smb"], errors="coerce")
    reg["hml"] = pd.to_numeric(reg["hml"], errors="coerce")
    reg["rf"] = pd.to_numeric(reg["rf"], errors="coerce")

    reg["excess"] = reg["LS"] - reg["rf"]
    reg = reg.dropna(subset=["excess", "mktrf", "smb", "hml"])

    X = sm.add_constant(reg[["mktrf", "smb", "hml"]])
    y = reg["excess"]

    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return reg, model

In [ ]:
reg_ew, model_ew = run_ff3(ew["LS"], ff)
reg_vw, model_vw = run_ff3(vw["LS"], ff)

print("ff3 for EW:", model_ew.summary())
print("\n\nff3 for VW:", model_vw.summary())

In [ ]:
perf_table["FF3 Alpha"] = [
    model_ew.params["const"],
    model_vw.params["const"],
]

perf_table["FF3 Alpha t-stat"] = [
    model_ew.tvalues["const"],
    model_vw.tvalues["const"],
]

perf_table["MKT Loading"] = [
    model_ew.params["mktrf"],
    model_vw.params["mktrf"],
]

perf_table["SMB Loading"] = [
    model_ew.params["smb"],
    model_vw.params["smb"],
]

perf_table["HML Loading"] = [
    model_ew.params["hml"],
    model_vw.params["hml"],
]

perf_table

In [ ]:
display_table = perf_table.copy()

pct_cols = ["Mean Monthly", "Annual Return", "Annual Vol", "Max Drawdown", "FF3 Alpha"]
for c in pct_cols:
    display_table[c] = display_table[c].map(lambda x: f"{x:.2%}")

num_cols = ["Sharpe", "t-stat", "FF3 Alpha t-stat", "MKT Loading", "SMB Loading", "HML Loading"]
for c in num_cols:
    display_table[c] = display_table[c].map(lambda x: f"{x:.2f}")

display_table

### 3.4 Interpretation of Results

#### **Risk-Adjusted Performance**

To evaluate whether the abnormal returns can be explained by standard risk factors, we estimate a Fama-French three-factor regression using HAC standard errors.

The **equal-weighted** long-short portfolio produces a **monthly FF3 alpha of 1.15%**, with a t-statistic of **3.37**, indicating that the strategy generates statistically significant abnormal returns even after controlling for market, size, and value factors.

The **value-weighted** strategy yields a **monthly alpha of 0.97%** with a t-statistic of **1.98**, which is marginally significant.

These results suggest that the asset growth signal captures return variation not fully explained by the traditional Fama-French factors.

#### **Factor Exposures**

The regression results also indicate a **strong positive loading on the HML (value) factor**, particularly for the value-weighted portfolio. This suggests that firms with lower asset growth tend to resemble value firms, while firms with higher asset growth tend to resemble growth firms.

The loadings on the market and SMB factors are relatively small, indicating that the strategy’s returns are not primarily driven by broad market exposure or size effects.

#### **Conclusion**

Overall, the results confirm that the asset growth signal generates significant long-short returns over the sample period. The anomaly appears stronger among smaller firms and remains significant after controlling for common risk factors.